# 05 — Supervised data preparation

This notebook:
- loads the SBS96 matrix and merged clinical data,
- defines the binary smoking target,
- creates train, validation, and test splits at patient level,
- prepares the feature matrices,
- saves the split tables for the model notebook.


In [103]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value in ["", "nan", "not reported", "unknown"]:
        return "Unknown"
    if "never" in value or "lifelong" in value or "non-smoker" in value or "nonsmoker" in value:
        return "Never"
    if "former" in value or "reformed" in value:
        return "Former"
    if "current" in value:
        return "Current"

    return "Unknown"

## 1. Define paths


In [104]:
sbs96_path = PROJECT_ROOT / "data" / "maf" / "output" / "SBS" /"TCGA_LUAD.SBS96.all"
clinical_path = PROJECT_ROOT / "data" / "clinical_exposure_merged.tsv"
split_output_dir = PROJECT_ROOT / "results" / "brf_split_binary"

split_output_dir.mkdir(parents=True, exist_ok=True)

display(
    pd.DataFrame({
        "file": ["SBS96 matrix", "clinical merged", "split output"],
        "path": [sbs96_path, clinical_path, split_output_dir],
    })
)

,file,path
0,SBS96 matrix,/Users/michaljendrusak/PycharmProjects/tcga-lu...
1,clinical merged,/Users/michaljendrusak/PycharmProjects/tcga-lu...
2,split output,/Users/michaljendrusak/PycharmProjects/tcga-lu...


## 2. Load the input tables


In [105]:
if sbs96_path is None:
    raise FileNotFoundError("The SBS96 matrix file was not found.")

sbs = pd.read_csv(sbs96_path, sep="\t", index_col=0)
clinical = pd.read_csv(clinical_path, sep="\t")

if sbs.shape[1] != 96 and sbs.shape[0] == 96:
    sbs = sbs.T

sbs_cols = list(sbs.columns)

display(
    pd.DataFrame({
        "table": ["SBS matrix", "clinical"],
        "rows": [sbs.shape[0], clinical.shape[0]],
        "columns": [sbs.shape[1], clinical.shape[1]],
    })
)

display(sbs.iloc[:5, :8])
display(clinical.head())

,table,rows,columns
0,SBS matrix,616,96
1,clinical,508,7


MutationType,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T
TCGA-05-4244-01A-01D-1105-08,5,5,4,2,3,0,1,1
TCGA-05-4249-01A-01D-1105-08,8,8,4,6,3,2,1,0
TCGA-05-4250-01A-01D-1105-08,7,1,9,11,0,2,3,2
TCGA-05-4382-01A-01D-1931-08,50,62,23,40,4,8,3,7
TCGA-05-4384-01A-01D-1753-08,5,1,3,1,0,0,2,0


,cases.submitter_id,exposures.tobacco_smoking_status,exposures.pack_years_smoked,demographic.age_at_index,demographic.gender,demographic.race,demographic.ethnicity
0,TCGA-62-A471,Current Reformed Smoker for < or = 15 yrs,30.0,64.0,male,white,not hispanic or latino
1,TCGA-67-3773,Current Reformed Smoker for > 15 yrs,NaN,84.0,female,white,not hispanic or latino
2,TCGA-NJ-A7XG,Current Reformed Smoker for > 15 yrs,NaN,49.0,male,black or african american,not hispanic or latino
3,TCGA-91-6848,Current Reformed Smoker for < or = 15 yrs,NaN,59.0,male,white,not hispanic or latino
4,TCGA-55-6986,Lifelong Non-Smoker,NaN,74.0,female,white,NaN


## 3. Merge SBS data with clinical data


In [106]:
sbs = sbs.copy()
sbs["Patient_ID"] = sbs.index.astype(str).str[:12]

sbs_patient = sbs.groupby("Patient_ID", as_index=False)[sbs_cols].mean()
sbs_patient["total_sbs"] = sbs_patient[sbs_cols].sum(axis=1)
sbs_patient = sbs_patient[sbs_patient["total_sbs"] > 0].drop(columns="total_sbs")

clinical = clinical.copy()
clinical = clinical.rename(columns={"cases.submitter_id": "Patient_ID"})
clinical["Patient_ID"] = clinical["Patient_ID"].astype(str).str[:12]

display(
    pd.DataFrame({
        "table": ["clinical", "sbs_patient"],
        "rows": [len(clinical), len(sbs_patient)],
        "unique_patient_ids": [
            clinical["Patient_ID"].nunique(),
            sbs_patient["Patient_ID"].nunique(),
        ],
    })
)

merged = pd.merge(clinical, sbs_patient, on="Patient_ID", how="inner")

display(
    pd.DataFrame({
        "table": ["merged"],
        "rows": [len(merged)],
        "unique_patient_ids": [merged["Patient_ID"].nunique()],
    })
)

display(
    merged[
        [
            "Patient_ID",
            "demographic.age_at_index",
            "demographic.gender",
            "exposures.tobacco_smoking_status",
        ]
    ].head()
)

,table,rows,unique_patient_ids
0,clinical,508,508
1,sbs_patient,557,557


,table,rows,unique_patient_ids
0,merged,493,493


,Patient_ID,demographic.age_at_index,demographic.gender,exposures.tobacco_smoking_status
0,TCGA-62-A471,64.0,male,Current Reformed Smoker for < or = 15 yrs
1,TCGA-67-3773,84.0,female,Current Reformed Smoker for > 15 yrs
2,TCGA-NJ-A7XG,49.0,male,Current Reformed Smoker for > 15 yrs
3,TCGA-91-6848,59.0,male,Current Reformed Smoker for < or = 15 yrs
4,TCGA-55-6986,74.0,female,Lifelong Non-Smoker


## 4. Define the binary target


In [107]:
merged["Smoking_3"] = merged["exposures.tobacco_smoking_status"].map(map_smoking_label)
merged = merged[merged["Smoking_3"].isin(["Never", "Former", "Current"])].copy()
merged["Smoking_Bin"] = (merged["Smoking_3"] != "Never").astype(int)

display(
    merged["Smoking_3"]
    .value_counts()
    .rename_axis("Smoking_3")
    .reset_index(name="n_rows")
)

display(
    merged["Smoking_Bin"]
    .value_counts()
    .rename(index={0: "Never", 1: "Ever"})
    .rename_axis("Smoking_Bin")
    .reset_index(name="n_rows")
)

,Smoking_3,n_rows
0,Former,305
1,Current,119
2,Never,69


,Smoking_Bin,n_rows
0,Ever,424
1,Never,69


## 5. Create patient-level train, validation, and test splits


In [108]:
RANDOM_STATE = 42

patient_labels = merged[["Patient_ID", "Smoking_Bin"]].drop_duplicates()

trainval_patients, test_patients = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=patient_labels["Smoking_Bin"],
)

train_patients, val_patients = train_test_split(
    trainval_patients,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=trainval_patients["Smoking_Bin"],
)

train_ids = set(train_patients["Patient_ID"])
val_ids = set(val_patients["Patient_ID"])
test_ids = set(test_patients["Patient_ID"])

display(
    pd.DataFrame({
        "check": ["train ∩ val", "train ∩ test", "val ∩ test"],
        "n_overlap": [
            len(train_ids & val_ids),
            len(train_ids & test_ids),
            len(val_ids & test_ids),
        ],
    })
)

,check,n_overlap
0,train ∩ val,0
1,train ∩ test,0
2,val ∩ test,0


## 6. Build row-level split tables and save them


In [109]:
train_df = merged[merged["Patient_ID"].isin(train_ids)].copy()
val_df = merged[merged["Patient_ID"].isin(val_ids)].copy()
test_df = merged[merged["Patient_ID"].isin(test_ids)].copy()

train_df.to_csv(split_output_dir / "train.csv", index=False)
val_df.to_csv(split_output_dir / "val.csv", index=False)
test_df.to_csv(split_output_dir / "test.csv", index=False)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_rows": [len(train_df), len(val_df), len(test_df)],
    "n_patients": [
        train_df["Patient_ID"].nunique(),
        val_df["Patient_ID"].nunique(),
        test_df["Patient_ID"].nunique(),
    ],
    "n_never": [
        (train_df["Smoking_Bin"] == 0).sum(),
        (val_df["Smoking_Bin"] == 0).sum(),
        (test_df["Smoking_Bin"] == 0).sum(),
    ],
    "n_ever": [
        (train_df["Smoking_Bin"] == 1).sum(),
        (val_df["Smoking_Bin"] == 1).sum(),
        (test_df["Smoking_Bin"] == 1).sum(),
    ],
})

split_summary.to_csv(split_output_dir / "split_summary.tsv", sep="\t", index=False)

display(split_summary)

,split,n_rows,n_patients,n_never,n_ever
0,train,315,315,44,271
1,validation,79,79,11,68
2,test,99,99,14,85


## 7. Prepare the feature matrices


In [110]:
def prepare_features(df, sbs_cols, age_fill=None):
    df = df.copy()

    df["demographic.age_at_index"] = pd.to_numeric(
        df["demographic.age_at_index"],
        errors="coerce"
    )

    df["demographic.gender"] = (
        df["demographic.gender"]
        .astype(str)
        .str.lower()
        .map({"male": 1, "female": 0})
    )

    if age_fill is None:
        age_fill = df["demographic.age_at_index"].median()

    df["demographic.age_at_index"] = df["demographic.age_at_index"].fillna(age_fill)

    X = df[sbs_cols + ["demographic.age_at_index", "demographic.gender"]].copy()
    X = X.rename(columns={"demographic.age_at_index": "age_years"})

    y = df["Smoking_Bin"].astype(int)

    return X, y, age_fill

In [111]:
X_train, y_train, age_fill = prepare_features(train_df, sbs_cols)
X_val, y_val, _ = prepare_features(val_df, sbs_cols, age_fill)
X_test, y_test, _ = prepare_features(test_df, sbs_cols, age_fill)

display(
    pd.DataFrame({
        "split": ["train", "validation", "test"],
        "n_rows": [len(X_train), len(X_val), len(X_test)],
        "n_features": [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
    })
)

display(
    pd.DataFrame({
        "imputation_value": ["age median"],
        "value": [age_fill],
    })
)

display(X_train.iloc[:5, :8].reset_index(drop=True))

,split,n_rows,n_features
0,train,315,98
1,validation,79,98
2,test,99,98


,imputation_value,value
0,age median,66.0


,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T
0,3.0,5.0,1.0,2.0,1.0,0.0,0.0,0.0
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,11.0,12.0,9.0,5.0,4.0,6.0,5.0,6.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,12.0,10.0,0.0,4.0,2.0,5.0,1.0,2.0
